In [11]:
pip install folium pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
!pip install folium pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
%%html
<div id="map" style="width: 100%; height: 500px; background: #f0f0f0; border-radius: 8px;"></div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
    // 1. Initialisation de la carte
    var map = L.map('map').setView([46.603354, 1.888334], 6);

    // 2. Ajout du fond de carte
    L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{z}.png', {
        maxZoom: 19,
        attribution: '&copy; CARTO'
    }).addTo(map);

    // 3. Chargement des données des régions françaises
    fetch('https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions.geojson')
        .then(response => response.json())
        .then(data => {
            var geojsonLayer = L.geoJson(data, {
                style: {
                    fillColor: '#3186cc',
                    weight: 1.5,
                    opacity: 1,
                    color: 'white',
                    fillOpacity: 0.4
                },
                onEachFeature: function(feature, layer) {
                    var nomRegion = feature.properties.nom;
                    layer.bindTooltip("<b>" + nomRegion + "</b>", { sticky: true });
                    
                    layer.on({
                        mouseover: function(e) {
                            e.target.setStyle({
                                fillColor: '#ff7800',
                                fillOpacity: 0.7
                            });
                        },
                        mouseout: function(e) {
                            geojsonLayer.resetStyle(e.target);
                        }
                    });
                }
            }).addTo(map);
        })
        .catch(err => console.error("Erreur de chargement :", err));

    // 4. Forcer le redimensionnement pour éviter l'affichage vide dans Jupyter
    setTimeout(function() {
        map.invalidateSize();
    }, 300);
</script>

In [ ]:
import os
import folium
from folium.plugins import MarkerCluster
import pandas as pd


def charger_donnees(chemin_fichier):
  """Charge le fichier CSV/TXT et convertit les coordonnées DMS en décimales."""
  print('Chargement et nettoyage des données...')
  df = pd.read_csv(chemin_fichier, sep=';', low_memory=False)

  def dms_to_dec(deg, mn, sc, ns_ew):
    try:
      dec = float(deg) + float(mn) / 60.0 + float(sc) / 3600.0
      if ns_ew in ['S', 'W']:
        dec = -dec
      return dec
    except Exception:
      return None

  df['LAT'] = df.apply(
      lambda r: dms_to_dec(
          r['COR_NB_DG_LAT'],
          r['COR_NB_MN_LAT'],
          r['COR_NB_SC_LAT'],
          r['COR_CD_NS_LAT'],
      ),
      axis=1,
  )
  df['LON'] = df.apply(
      lambda r: dms_to_dec(
          r['COR_NB_DG_LON'],
          r['COR_NB_MN_LON'],
          r['COR_NB_SC_LON'],
          r['COR_CD_EW_LON'],
      ),
      axis=1,
  )

  return df.dropna(subset=['LAT', 'LON'])


def obtenir_couleur_operateur(row):
  """Associe une couleur selon le code TPO_ID (converti proprement en entier)."""
  val = row.get('TPO_ID')
  if pd.isna(val):
    return 'black', 'Multi-opérateurs / Non renseigné'

  try:
    tpo = int(float(val))  # Convertit 32.0 -> 32
  except Exception:
    return 'black', 'Multi-opérateurs'

  # Répartition basée sur les codes TPO_ID présents dans votre fichier
  if tpo == 74:
    return 'orange', 'Orange (TPO 74)'
  elif tpo == 32:
    return 'red', 'SFR (TPO 32)'
  elif tpo == 29:
    return 'brown', 'Free Mobile (TPO 29)'
  elif tpo == 72:
    return 'blue', 'Bouygues Telecom (TPO 72)'
  else:
    return 'black', f'Multi-opérateurs / Autre (TPO {tpo})'


def creer_carte(df):
  """Initialise la carte OpenStreetMap et ajoute les repères et antennes."""
  carte = folium.Map(
      location=[46.603354, 1.888334], zoom_start=6, tiles='OpenStreetMap'
  )

  villes = [
      ('Paris', 48.8566, 2.3522),
      ('Lyon', 45.7640, 4.8357),
      ('Marseille', 43.2965, 5.3698),
      ('Toulouse', 43.6047, 1.4442),
      ('Bordeaux', 44.8378, -0.5792),
      ('Lille', 50.6292, 3.0573),
      ('Nantes', 47.2184, -1.5536),
      ('Strasbourg', 48.5734, 7.7521),
  ]
  for nom, lat, lon in villes:
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=(
                '<div style="font-size: 11px; font-weight: bold; color: #333;'
                ' background: rgba(255,255,255,0.8); padding: 2px 5px;'
                f' border-radius: 3px; border: 1px solid #ccc;">{nom}</div>'
            )
        ),
    ).add_to(carte)

  print(f'Ajout de {len(df)} antennes sur la carte...')
  cluster = MarkerCluster().add_to(carte)

  for _, row in df.iterrows():
    color, op_name = obtenir_couleur_operateur(row)

    popup_text = f"""
        <b>Support ID:</b> {row['SUP_ID']}<br>
        <b>Commune (INSEE):</b> {row['COM_CD_INSEE']}<br>
        <b>Catégorie:</b> {op_name}
        """

    folium.CircleMarker(
        location=[row['LAT'], row['LON']],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300),
    ).add_to(cluster)

  legende_html = """
    <div style="
        position: fixed; bottom: 50px; left: 50px; width: 220px; height: 160px; 
        background-color: white; z-index:9999; font-size:13px;
        border:2px solid grey; border-radius: 5px; padding: 10px;
        box-shadow: 0 0 15px rgba(0,0,0,0.2);">
        <b>Légende - Opérateurs</b><br><br>
        <i style="background:blue; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Bouygues Telecom<br>
        <i style="background:orange; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Orange<br>
        <i style="background:red; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> SFR<br>
        <i style="background:brown; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Free Mobile<br>
        <i style="background:black; width:10px; height:10px; float:left; margin-right:8px; border-radius:50%; margin-top:4px;"></i> Multi-opérateurs / Autre<br>
    </div>
    """
  carte.get_root().html.add_child(folium.Element(legende_html))

  return carte


# --- EXÉCUTION PRINCIPALE ---
chemin = 'C:/Users/guibeleo/Desktop/SAE15-D3/data/sup_support.txt'
sortie_html = 'C:/Users/guibeleo/Desktop/SAE15-D3/carte_antennes_france.html'

if os.path.exists(chemin):
  df_antennes = charger_donnees(chemin)
  ma_carte = creer_carte(df_antennes)
  ma_carte.save(sortie_html)
  print(f'Carte enregistrée dans : {sortie_html}')
  display(ma_carte)
else:
  print(f'Erreur : Fichier introuvable à l’emplacement {chemin}')